In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
ngpu=1
ngf,nc = 3,3
ndf = 64

transform = transforms.Compose([
    transforms.Resize((450,450)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [5]:
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [6]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 1)
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [7]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 207MB/s] 


In [8]:
fsc_submission=pandas.read_csv("/kaggle/input/ai-vs-human-generated-dataset/test.csv")['id']

In [9]:
sub, z = {}, 0

for j in ['1.5027252629806753e-05 21.pth']:

    z_add,z_total=0,0
    EFF_NET.load_state_dict(torch.load(f"/kaggle/input/dataset-human-vs-ai/{j}", 
                                       weights_only=False,
                                       map_location=torch.device('cpu')))
    
    for i,j in zip(fsc_submission,range(len(fsc_submission))):
        
        img = transform(Image.open(f'/kaggle/input/d/mafiosoquasar/detect-ai-vs-human-generated-images/test/{i.split("/")[1]}')).reshape((1, 3, 450, 450)).float().to(device)    
        output = EFF_NET(img).sigmoid().cpu().detach().numpy()[0][0]
        
        if output==0:
            z_add+=1
        z_total+=1
        #fsc_submission.loc[i]['label']=output
        if i not in sub.keys():
            if output==0:
                sub[i]={0:1,
                       1:0}
            else:
                sub[i]={0:0,
                       1:1}
        else:
            if output==0:
                sub[i][0]+=1
            else:
                sub[i][1]+=1

        z+=1
        print(z)
    print(z_total,z_add)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201 0


In [28]:
for i in fsc_submission:
    sub[i]=list(sub[i].values())

AttributeError: 'list' object has no attribute 'values'

In [25]:
z_add,z_total=0,0

label=[]
for i in fsc_submission:
    if sub[i][0]>=6:
        label.append(0)
        z_add+=1
    else:
        label.append(1)
        
    z_total+=1
print(z_total,z_add)

submission=pandas.DataFrame({'id' : fsc_submission, 
                             'label' : label})
submission

test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg
test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg
test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg
test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg
test_data_v2/a16495c578b7494683805484ca27cf9f.jpg
test_data_v2/d08826f7ccab45c8935d8df5524b2869.jpg
test_data_v2/8ba52123cc7b4e3aa90e3947734197e0.jpg
test_data_v2/b107ac0040284f1aace7a6714bf152f7.jpg
test_data_v2/601c646d626d46da8c5fac6653efcfb8.jpg
test_data_v2/ef29ead63754441b82b56c1a22082fdf.jpg
test_data_v2/f166b995659946e7bcaa25823890d909.jpg
test_data_v2/bc1fe6d5527246eca53bbfec751d4020.jpg
test_data_v2/72db1bbd6a0d46a8b72b240cacfb1794.jpg
test_data_v2/8bb829efccb549178cae75a2c6b7780f.jpg
test_data_v2/eb476c70a9ce427d810a265c492888d4.jpg
test_data_v2/ae9f3fd741dd45f0ae94fce7d0b3ab6b.jpg
test_data_v2/a42e0b2c275242feaf8618d0c74fc419.jpg
test_data_v2/524eaa42b4324cc2a66baec78e8e6984.jpg
test_data_v2/57e48cca910d41d4886ffb48ec992d95.jpg
test_data_v2/f79a9aa7d7a1499ca22a1b5c1ee7ec69.jpg


,id,label
0,test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg,1
1,test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg,1
2,test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg,1
3,test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg,1
4,test_data_v2/a16495c578b7494683805484ca27cf9f.jpg,1
...,...,...
196,test_data_v2/5b056a650cba443397e9cb86a8312511.jpg,1
197,test_data_v2/9e6306978ea249368258757ad26d1f93.jpg,1
198,test_data_v2/a98d8ff2747946a980bcccebd7612552.jpg,1
199,test_data_v2/61fc9030746d452590802c5511345b35.jpg,1


In [27]:
pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)